###  第一阶段准备数据

In [18]:
#构建词表
import re
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
 "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
 "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text= f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])


#还需要分割
# #text = "Hello, world. This, is a test."
# text = "Hello, world. Is this-- a test?"
# #text = raw_text
# #result = re.split(r'(\s)',text)
# #通过正则表达式把文本分割成特定的单元(TOKENIZED TEXT)
# result = re.split(r'([,.]|\s)',text)
# result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
# print(result)

pattern = re.compile(r"""(--|[,.:;?_!"()']|\s+)""")
tokens = [part for part in pattern.split(raw_text) if part.strip()]

# 每个不同的 token 对应一个整数 ID
vocab = {
    token: token_id
    for token_id, token in enumerate(sorted(set(tokens)))
}

print("字符数:", len(raw_text))
print("Token 数:", len(tokens))
print("词表大小:", len(vocab))
print("前 30 个 token:", tokens[:30])
print("前 10 个词表项:", list(vocab.items())[:10])


Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 
字符数: 20479
Token 数: 4690
词表大小: 1130
前 30 个 token: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
前 10 个词表项: [('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9)]


In [20]:
#添加特殊的词元
for token in ("<|endoftext|>", "<|unk|>"):
    vocab[token]=len(vocab)


In [21]:
print(len(vocab))                  # 1132
print(vocab["<|endoftext|>"])      # 1130
print(vocab["<|unk|>"])  

1132
1130
1131


In [22]:
id_to_token = {token_id: token for token, token_id in vocab.items()}

print(vocab["<|endoftext|>"])   # token → ID：1130
print(id_to_token[1130])        # ID → token：<|endoftext|>
print(id_to_token[1131])        # <|unk|>

1130
<|endoftext|>
<|unk|>


In [24]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


In [28]:
tokenizer = tiktoken.get_encoding("gpt2")

In [26]:
print(tiktoken.list_encoding_names())

['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']


In [34]:
import torch

print(torch.__version__)

vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)


2.7.1+cpu
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [35]:
from torch import nn

In [38]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query=nn.Parameter(torch.rand(d_in,d_out))
        self.W_key=nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        scores = queries @ keys.T
        weights = torch.softmax(
            scores/ keys.shape[-1]**0.5,
            dim = -1,
        )
        z = weights @ values
        return z

                                
        

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out，qkv_bias=False):
        super().__init__()
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        scores = queries @ keys.T
        weights = torch.softmax(
            scores/ keys.shape[-1]**0.5,
            dim = -1,
        )
        z = weights @ values
        return z


In [ ]:
class CasualAttention(nn.Module):
    def __init__(self, d_int, d_out, context_length, droput, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_lenght),
                       diagonal = 1
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            ann_scores/ keys.shape[-1]**0.5,
            dim = -1,
        )
        attn_weights = self.dropout(attn_weights)
        z = attn_weights @ values
        return z

        

In [ ]:
#这个只是一个Wrapper不是真正的Multiple Header
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_lenght, dropout, qkv_bias
            )
             for _ in range( num_heads)]
        )
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)

In [ ]:
#真正的MultiAttention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        #d_out 必须要能被num_heads整除
        assert(d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_lenght),
                       diagonal = 1
        )

    def forward(self,x):
        b, num_tokens, d_in = x.shape #bacth , token , dim
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_token, self.num_heads,self.head_dim)
        values = values.view(b, num_token, self.num_heads,self.head_dim)
        queries = queries.view(b, num_token, self.num_heads,self.head_dim)

        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2, 3) 
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        
        attn_scores.masked_fill_(
            maks_bool, -torch.inf
        )
        attn_weights = torch.softmax(
            ann_scores/ keys.shape[-1]**0.5,
            dim = -1,
        )
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(
         b, num_tokens, self.d_out
         )
        context_vec = self.out_proj(context_vec)

        return context_vec

        
        
        
        
            
            


        